In [ ]:
import shutil
import os

import matplotlib.image as mpimg
import matplotlib.pyplot as plt

PATH = "path_2"
YOLO_MODEL = "v4"

labels_path = f"data/{PATH}/{YOLO_MODEL}/labels/*.txt"
images_path = f"data/{PATH}/{YOLO_MODEL}/*.jpg"
save_results_path = f"results/{PATH}/{YOLO_MODEL}"

if os.path.exists(f"results/{PATH}/{YOLO_MODEL}"):
    shutil.rmtree(f"results/{PATH}/{YOLO_MODEL}")
    print("Folder sucessfully removed.")

os.makedirs(f"results/{PATH}/{YOLO_MODEL}")

# Image resolution settings
# Note: The second 'scale' assignment overrides the first in Python
width_scale = 1280  # in pixels
height_scale = 720  # in pixels

# Real-world dimensions
path_width_cm = 1000  # path width in centimeters

In [ ]:
def calculations(coordinates) -> list:

    # Extract basic dimensions
    center_x, center_y, width, height = coordinates["1"][0][:4]

    # Define the boundaries
    left   = center_x - width / 2.0
    right  = center_x + width / 2.0
    top    = center_y - height / 2.0
    bottom = center_y + height / 2.0

    # Generate all 9 candidate vertices
    candidates = [
    (left, top),    (center_x, top),    (right, top),    # Top row
    (left, center_y), (center_x, center_y), (right, center_y), # Mid row
    (left, bottom), (center_x, bottom), (right, bottom)  # Bottom row
    ]
    
    if coordinates["type"] == "std":
        car_center_x, car_center_y = coordinates["0"][0][:2]
    else:
        car_x1, car_y1 = coordinates["0"][0][0], coordinates["0"][0][1]
        car_x2, car_y2 = coordinates["0"][0][2], coordinates["0"][0][3]

        # Vertex 4 (top left), Vertex 3 (top right)
        car_x4, car_y4 = coordinates["0"][0][6], coordinates["0"][0][7]
        car_x3, car_y3 = coordinates["0"][0][4], coordinates["0"][0][5]
        
        car_center_x = (car_x1 + car_x2 + car_x4 + car_x3) / 4.0
        car_center_y = (car_y1 + car_y2 + car_y4 + car_y3) / 4.0

    # Find the point with the minimum Euclidean distance
    target_x, target_y = min(
        candidates, 
        key=lambda p: (p[0] - car_center_x)**2 + (p[1] - car_center_y)**2
    )

    # Scale factor (pixels to cm) - using the box width as reference
    pixel_width = width # simplified from your distance formula
    current_scale = path_width_cm / pixel_width

    # Calculate error in centimeters relative to the closest vertex
    x_error = current_scale * (target_x - car_center_x)
    y_error = current_scale * (target_y - car_center_y)

    # Store squared error for MSE (EQM) calculations
    error_log.append(x_error**2 + y_error**2)

    return [target_x, target_y, car_center_x, car_center_y]

In [ ]:
import glob

error_log = list()

for n, (label_path, img_path) in enumerate(zip(glob.glob(labels_path), glob.glob(images_path))):
    # Dictionary to store detected coordinates and detection format
    coordinates = {"0": [], "1": [], "type": str}

    with open(label_path, "r") as f:
        for line in f:
            data = line.removesuffix("\n").split()
            cls = data[0]
            # Appending normalized coordinates to the respective class
            coordinates[cls].append(list(map(lambda x: float(x), data[1:])))

            # Determine if format is OBB (xyxy...) or Standard Bounding Box (xywhn)
            coordinates["type"] = "obb" if len(data[1:]) > 5 else "std"

    try:
        target_x, target_y, car_center_x, car_center_y = calculations(coordinates)

        plt.figure()
        plt.plot(
            [val * width_scale for val in [target_x, car_center_x]],
            [val * height_scale for val in [target_y, car_center_y]],
            c="r",
        )

        img = mpimg.imread(img_path)
        plt.imshow(img)
        plt.axis('off')
        plt.savefig(save_results_path + f"/{n}.png", dpi=480)
        plt.show()
        
    except IndexError:
        print('Image does not cointain objects of 2 classes. Moving to next...')

In [ ]:
# Calculating accuracy metrics
mse = sum(error_log) / len(error_log)  # Mean Squared Error
rmse = mse ** (1 / 2)  # Root Mean Squared Error

print(f"Mean Squared Error (MSE) = {mse:.4f}")
print(f"Root Mean Squared Error (RMSE) = {rmse:.4f}")

print("Note: Measurements are in CM")